# Day E1 — EHR 데이터는 누가, 왜 남긴 기록인가: 임상 질문을 연구 질문으로

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/E-ehr/day-E1.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/E-ehr/day-E1.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 다섯 칸 질문표를 코드로 만들어 두기

질문을 dict로 적어 두면 나중에 코호트 코드를 짤 때 그대로 참고할 수 있고, 빈칸이 있으면 바로 보입니다.

In [ ]:
question = {
    "대상": "18세 이상, 첫 ICU 입실, 입실 6시간 안에 저혈압(MAP < 65)이 있었던 환자",
    "노출": "저혈압 발생 후 3시간 안에 승압제 시작",
    "비교": "같은 조건에서 3시간 안에 승압제를 시작하지 않은 환자",
    "결과": "ICU 입실 후 28일 사망",
    "시점": "",          # 아직 정하지 못함
}
missing = [k for k, v in question.items() if not v]
print("빈칸:", missing)
print(f"대상: {question['대상']}")
print(f"질문: [{question['노출']}]은 [{question['비교']}]에 비해 "
      f"[{question['결과']}]과 어떤 연관이 있는가?")

### 예시 2 — 건수는 늘었는데 비율은 줄었다

가상 병원에서 어떤 장치를 쓴 환자 수를 해마다 세어 봅니다.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(1)
rows = []
for year in range(2015, 2021):
    n_adm = 800 + 120 * (year - 2015)            # 해마다 입원 환자가 늘어난다
    p_use = 0.30 - 0.02 * (year - 2015)          # 장치를 쓰는 '비율'은 줄어든다
    used = rng.random(n_adm) < p_use
    rows.append({"year": year, "admissions": n_adm, "device_used": used.sum()})
trend = pd.DataFrame(rows)
trend["rate_%"] = (100 * trend["device_used"] / trend["admissions"]).round(1)
print(trend.to_string(index=False))

### 예시 3 — "검사를 많이 했다"는 것 자체가 신호다

기록에는 없는 "실제 중증도"가 검사 횟수와 사망을 동시에 움직이는 가상 상황입니다.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)
n = 2000
severity = rng.normal(0, 1, n)                     # 기록에는 없는 '실제 중증도'
n_labs = rng.poisson(np.exp(1.5 + 0.6 * severity)) # 아픈 환자일수록 검사를 자주 한다
p_death = 1 / (1 + np.exp(-(-2.5 + 1.2 * severity)))
died = rng.random(n) < p_death
df = pd.DataFrame({"n_labs": n_labs, "died": died})
df["lab_group"] = pd.qcut(df["n_labs"], 4, labels=["Q1(적음)", "Q2", "Q3", "Q4(많음)"])
print(df.groupby("lab_group", observed=True)["died"].agg(["count", "mean"]).round(3))